In [20]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy.stats import chi2_contingency

In [5]:
data1 = "D:/Data Analysis/Data/statistical_hypothesis/cookie_cats.csv"
df = pd.read_csv(data1)
type(df)

pandas.DataFrame

In [6]:
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [9]:
effect_size = sms.proportion_effectsize(0.20, 0.19)
effect_size

np.float64(0.025241594409087353)

In [12]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power = 0.8,
    alpha = 0.05,
    ratio = 1
)
required_n = ceil(required_n)
print(required_n)   

24638


Сумарно нам буде потрібно 49276 користувачів.

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [15]:
retention_stats = df.groupby('version')['retention_7'].mean()

print("Середнє утримання на 7-й день за версіями:")
print(retention_stats)

Середнє утримання на 7-й день за версіями:
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


Краща версія з воротами на 30 рівні, з конверсією 19%.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [21]:
results_30 = df[df['version'] == 'gate_30']['retention_7']
results_40 = df[df['version'] == 'gate_40']['retention_7']

successes = [results_30.sum(), results_40.sum()]
nobs = [results_30.count(), results_40.count()]

z_stat, pval = proportions_ztest(successes, nobs = nobs)
(lower_30, lower_40), (upper_30, upper_40) = proportion_confint(successes, nobs = nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи gate_30: [{lower_30:.3f}, {upper_30:.3f}]')
print(f'Довірчий інтервал 95% для групи gate_40: [{lower_40:.3f}, {upper_40:.3f}]')

z statistic: 3.16
p-value: 0.002
Довірчий інтервал 95% для групи gate_30: [0.187, 0.194]
Довірчий інтервал 95% для групи gate_40: [0.178, 0.186]


1. Так, різниця статистично значуща. Оскільки p-value менше за 0.05.

2. Не перетинаються, але лежать доволі близько. Свідчить про те що різниця не дуже сильна, але вона є.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [25]:
crosstab = pd.crosstab(df['version'], df['retention_7'])

chi2, p_value, dof, expected = chi2_contingency(crosstab)

print(f"Таблиця спряженості:\n{crosstab}\n")
print(f"p-value: {p_value:.5f}")
print(f"Статистика Хі-квадрат: {chi2:.4f}")

Таблиця спряженості:
retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279

p-value: 0.00160
Статистика Хі-квадрат: 9.9591


1.
  
 H0 : повернення однакове у двох варіантах.
   
   Ha : є різниця між двома варінтами.

Висновок: 

Р-value = 0.0016, що менше за 0.05, що підтверджує статистично значущу різницю. 

Варіант з воротами на 30 рівні є кращим.